# Cost Engineering- Every Token Has a Price

**Module 7 · Lesson 7.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Token Counting - Know What You Spend
- Cheap-First-Escalate - The 90/10 Strategy
- Semantic Caching - Never Pay Twice
- Prompt Compression - Fewer Tokens, Same Info
- Cost Observability - Helicone, Langfuse, Provider Dashboards
- Token Optimization - Six Techniques That Compound to 50-80% Savings

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install tiktoken numpy google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The First Invoice: 17x Over Budget

Your chatbot shipped last month. It works - users love it. Then finance forwards the provider invoice with one line highlighted:

**📝 `invoice_review.txt`** - *Intro*

In [ ]:
# Output: the budget meeting, reconstructed
#
# Your estimate (before launch):
#   500 in + 300 out tokens x 200K conversations x premium model  ~= ₹30,000 / month
#
# The invoice:                                                    ₹5,10,000 / month
#
# Where the 17x came from (the post-mortem):
#   x2.1  multi-turn context - turn 5 re-sends turns 1-4; input grows ~n(n+1)/2
#   x1.4  the "quality check" LLM-judge you added - a full model call, never metered
#   x1.15 retries on timeouts - each one re-sends the WHOLE conversation
#   x1.6  every request on the premium model - including "what are your hours?"
#   x1.3  a 2,000-token system prompt, re-billed on every single request
#
# Nobody wrote a bug. Every factor was a default you never questioned.

Cost engineering is the discipline of questioning those defaults. This lesson builds the four core techniques (measure, route, cache, compress), then the production layer (observability, stacked discounts, hard budgets), and finally the India-specific line items your finance team will actually see. The target is real: this exact bill ends the lesson at ~₹30,000 - the original estimate - with the same quality.

> 📦 **Info**
>
> 💰 Before you start: how to read a token bill
> - **Billed tokens come from the API response** - `response.usage` (input_tokens / output_tokens) is the invoice truth. Client-side counters like `tiktoken` are OpenAI's tokenizer only: fine for OpenAI estimates and format comparisons, off by 15-20%+ for Claude and Gemini (use their `count_tokens` endpoints or reported usage).
> - **Output tokens cost 2-5x input tokens** on every major provider - the expensive side of the bill is what the model SAYS, not what you send.
> - **Prices move monthly.** Every ₹/$ figure in this lesson is stamped with its capture date in the research notes; re-check before you architect around one.
> - **Prereqs:** the 7.1 cost ledger and typed errors, 7.2's usage-in-streams (and the cancelled-stream billing blind spot). Same three env-var keys.

Sixty-second warm-up - three cards from the lessons this one stands on. Tap each to flip it.

> 💰 **Analogy**
>
> **Running an unoptimized AI app is like taking a taxi everywhere.** Taxi 1 km: ₹200. Auto: ₹30. Bus: ₹10. Walking: ₹0. Cost engineering = choosing the right transport for each trip. **Same destination. 95% cheaper.**
> **Where this analogy breaks down:** transport fares are posted on the door and change yearly; LLM fares change monthly and differ per DIRECTION - the meter runs ~2-5x faster on the return trip (output tokens). And unlike a taxi, a cheaper ride that takes you to the wrong address costs you the fare TWICE - that is the cost paradox of Step 8.

> 💡 **Info**
>
> ⚠️ Common misconception
> You might think the cheapest model per token is the cheapest model, full stop. It is not: a budget model that fails 60% of tasks and retries in loops can cost MORE per successful output than the premium model it replaced - and switching providers resets your prompt caches on top. The honest metric is **cost-per-successful-output (CPSO)**: total spend divided by outputs that needed no correction. Step 8's animation makes the paradox visible.

## Measure and Cut: The Four Core Techniques

---

## 🎯 Concept 1: Token Counting - Know What You Spend

### Token Counting - Know What You Spend

You can't optimize what you don't measure.

The cold-open invoice was a surprise because nobody was reading the meter. Before any optimization, we install the meter: count what every call costs, project it to a month, and learn which numbers to trust.

> 📈 **Analogy**
>
> **A utility meter you actually own.** You cannot dispute an electricity bill if your only meter is a neighbour's lookalike unit - and that is what `tiktoken` is for Claude and Gemini: OpenAI's meter strapped onto someone else's wiring, off by 15-20%. The billing-grade meter is the provider's own reading (`response.usage`). Use the lookalike for estimates and format comparisons; use the real meter for money.

**📝 `01_token_counter.py`** - *tiktoken*

In [ ]:
# TOKEN COUNTING & COST TRACKING
import tiktoken  # OpenAI's tokenizer: fine for OpenAI + format comparisons.
# For Claude/Gemini BILLS, use API-reported response.usage (or their count_tokens
# endpoints) - tiktoken misreads their tokenizers by 15-20%+. Estimates only here.
class CostTracker:
    PRICING = {  # $/M tokens (input, output) - captured 2026-07-02; prices move monthly
        "gemini-2.5-flash-lite":(0.10,0.40), "gpt-5.4-mini":(0.75,4.50),
        "gpt-5.4":(2.50,15.0), "claude-sonnet-4-6":(3.0,15.0)}
    def __init__(self):
        self.enc=tiktoken.get_encoding("cl100k_base"); self.total_cost=0; self.calls=[]
    def count(self,t): return len(self.enc.encode(t))
    def log(self,model,prompt,resp):
        ti,to=self.count(prompt),self.count(resp)
        if model not in self.PRICING: raise KeyError(f"no price for {model} - add it")  # fail loud, never $0-or-guess
        p=self.PRICING[model]; c=(ti*p[0]+to*p[1])/1e6
        self.total_cost+=c; self.calls.append({"model":model,"cost":c}); return c
    def report(self):
        print(f"  {len(self.calls)} calls | Total: ${self.total_cost:.6f}")
        print(f"  Daily (10K): ${self.total_cost/max(len(self.calls),1)*10000:.2f}")
        print(f"  Monthly:     ${self.total_cost/max(len(self.calls),1)*300000:.2f}")

t=CostTracker()
t.log("gemini-2.5-flash-lite","What is Python?","Python is a language."*5)
t.log("gpt-5.4","Explain microservices..."*10,"Microservices..."*50)
t.log("claude-sonnet-4-6","Write a plan..."*20,"Executive summary..."*100)
t.report()

# Output: (representative run)
#   3 calls | Total: $0.004273
#   Daily (10K): $14.24
#   Monthly:     $427.29
# The premium call cost ~30x the flash-lite call for the same round trip -
# and remember: this is the TOKEN bill only (~30-40% of true spend).

- `PRICING` - the tariff card, date-stamped. Per-model (input, output) pairs in $/M; the KeyError guard means an unknown model stops the ledger instead of silently mispricing it (the 7.1 bug, done right).

- `count()` - the estimate meter. tiktoken encodes text exactly the way OpenAI bills it; for Claude/Gemini it is an approximation - swap in `response.usage` when you wire this into real calls.

- `log()` - one receipt per call: count both directions, price them separately (output is the expensive lane), keep the running total.

- `report()` - the projection finance wants: per-call average scaled to 10K/day and a month. Honest caveat printed with it: tokens are only the tip of the bill.

Read it as: *meter first, tariff card second, projection last - and never price what you have not metered.*

#### 💡 What just happened

⚡ What just happened?`CostTracker` counts every token, calculates cost per call, projects daily/monthly. A GPT-4o call costs 25x more than the same call on Gemini Flash. **Rule: measure before optimizing.**

- **Scene 1 (the waterfall):** the ₹5,00,000 bill from the cold open. Do: toggle each layer - routing, caching, compression, batch - and watch the bar fall down the waterfall. Notice: the scratchpad shows the multipliers MULTIPLYING, not adding - that is why four modest layers reach ~94%.

- **Scene 2 (the iceberg):** the token bill is the tip. Reveal the submerged blocks - infrastructure, evaluation, guardrails, retries - and watch the REAL total reach ~2.5-3x the API line. Budgeting tokens x1.1 is the classic mistake this scene un-teaches.

Controls: Play / Reset / Speed (Slow / Normal / Fast); layer toggles; scene buttons or arrow keys.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Cheap-First-Escalate - The 90/10 Strategy

### Cheap-First-Escalate - The 90/10 Strategy

Try cheapest model first. Escalate only if quality is insufficient.

The meter shows a 30x price gap between your cheapest and dearest models for the same round trip. The first structural saving: stop sending every query to the specialist. Send it to the GP first - and be honest about what the receptionist costs.

> 🩺 **Analogy**
>
> **A clinic with a GP, a specialist, and a salaried receptionist.** Most complaints resolve at the GP (the cheap model). The receptionist (the LLM judge) decides who gets referred - quick, but not free: she draws a salary on EVERY visit, referred or not. A clinic that brags about specialist savings while hiding the receptionist's payroll is cooking its books - which is precisely what the un-metered judge did to the cold-open invoice.

Cheap-First-Escalate

**📝 `02_escalation.py`** - *Production*

In [ ]:
# CHEAP-FIRST-ESCALATE - honest version: it really escalates, and the judge is metered
from google import genai  # google-generativeai reached EOL 2025-11-30
client = genai.Client()      # reads GOOGLE_API_KEY

CHEAP, PREMIUM, JUDGE = "gemini-2.5-flash-lite", "gemini-2.5-pro", "gemini-2.5-flash-lite"

class CheapFirstRouter:
    def __init__(self):
        self.stats={"cheap":0,"escalated":0,"judge_calls":0}  # the judge is a COST, count it
    def _ask(self, model, prompt):
        return client.models.generate_content(model=model, contents=prompt).text.strip()
    def _good_enough(self, q, r):
        self.stats["judge_calls"] += 1
        verdict = self._ask(JUDGE, f"Rate 1-10: does this answer the query?\nQ:{q[:200]}\nA:{r[:300]}\nReturn ONLY the number.")
        try: return int(verdict.split()[0]) >= 7
        except ValueError: return True   # unparseable judge = don't burn premium money on noise
    def generate(self, query):
        r = self._ask(CHEAP, query)
        if self._good_enough(query, r):
            self.stats["cheap"] += 1; return r, CHEAP
        self.stats["escalated"] += 1
        return self._ask(PREMIUM, query), PREMIUM   # a REAL escalation - the previous version re-asked the cheap model

router = CheapFirstRouter()
for q in ["What is 2+2?", "Translate hello to Hindi", "Design a distributed system for 10M users"]:
    ans, model = router.generate(q)
    print(f"  [{model:22s}] '{q[:42]}' -> {ans[:48]}...")
print(f"\n  {router.stats['cheap']}/{router.stats['cheap']+router.stats['escalated']} served cheap, "
      f"{router.stats['judge_calls']} judge calls (meter them!)")

# Output: (representative run)
#   [gemini-2.5-flash-lite ] 'What is 2+2?' -> 4...
#   [gemini-2.5-flash-lite ] 'Translate hello to Hindi' -> नमस्ते (namaste)...
#   [gemini-2.5-pro        ] 'Design a distributed system for 10M user' -> A system at this scale needs...
#
#   2/3 served cheap, 3 judge calls (meter them!)
# Honest math: every request costs cheap + judge; escalations add premium on top.
# The judge only pays for itself when (escapes prevented x premium price) > judge spend.

- `CHEAP / PREMIUM / JUDGE` - the triage roster: a GP, a specialist, and the receptionist who decides. The receptionist draws a salary too - `judge_calls` is on the ledger because un-metered judges are how the cold-open invoice grew a silent x1.4.

- `_good_enough()` - an LLM-as-judge with a numeric contract ("Return ONLY the number") and a defensive parse: if the judge rambles, we accept the cheap answer rather than burn premium money on noise.

- the escalation line - goes to `PREMIUM`. The previous version of this lesson re-asked the SAME cheap model to "provide a better answer" - a placebo escalation that matched neither its own diagram nor its savings claims.

- threshold `>= 7` - your quality dial: raise it and more traffic escalates (costlier, better); lower it and savings rise with risk. Tune it on YOUR eval set, not this demo's three queries.

Read it as: *GP first, specialist on referral - and the receptionist's salary is on the books.*

#### 💡 What just happened

⚡ What just happened? Simple queries passed the quality gate and were served cheap; the complex one got a REAL premium escalation. Typical production shape: 70-90% of traffic passes the cheap tier. But do the honest math - every request now costs cheap + judge, so savings = (premium price - cheap - judge) on the passed share. The router you built in Lesson 7.1 supplies the fallback machinery underneath this. **Measure the judge, or your savings slide is fiction.**

- **Scene 1 (the escalator):** queries hit the cheap model; the judge gate scores each answer; passes serve cheap (teal), fails escalate to premium (amber). Notice: THREE ledgers - "everything premium", "cheap-first including the judge's own cost", and savings. The judge's meter runs on every request.

- **Scene 2 (the cost paradox):** a premium model at 95% success races a budget model at 40% with visible retry loops. Notice: cost-per-token says the budget model wins; total spend and cost-per-successful-output say otherwise.

Controls: Play / Reset / Speed (Slow / Normal / Fast); scene buttons or arrow keys.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Semantic Caching - Never Pay Twice

### Semantic Caching - Never Pay Twice

Similar question asked before? Return cached answer. Zero LLM cost.

Routing cut the price of answers you still had to generate. The bigger win is not generating at all: most products hear the same questions in a hundred phrasings, and an answer you have already paid for once should be free the second time.

> 📦 **Analogy**
>
> **Semantic cache = restaurant thali counter.** Cook biryani fresh: 45 min (LLM call). Pre-made batch: instant. "What is ML?" matches "What is machine learning?" in the cache - same answer, free.
> **Where this analogy breaks down:** at a thali counter you can SEE you got the wrong dish; a semantic cache serves the wrong answer with total confidence - "What is MLOps?" can match "What is ML?" at 0.86 similarity and nobody notices. The counter also never goes stale visibly: cached answers outlive price changes, product updates, and policy edits unless you expire them.

**📝 `03_semantic_cache.py`** - *Cache*

In [ ]:
# SEMANTIC CACHING - Match by meaning, not exact text
from google import genai  # the current SDK (google-generativeai reached EOL 2025-11-30)
import numpy as np
client = genai.Client()

class SemanticCache:
    def __init__(self,threshold=0.90):  # tune on real traffic: higher = safer, fewer hits
        self.threshold=threshold; self.cache=[]; self.stats={"hits":0,"misses":0}
    def _embed(self,t):
        r = client.models.embed_content(model="gemini-embedding-001", contents=t)
        return np.array(r.embeddings[0].values)
    def query(self,prompt):
        qe=self._embed(prompt)
        for e in self.cache:
            sim=np.dot(qe,e["emb"])/(np.linalg.norm(qe)*np.linalg.norm(e["emb"]))
            if sim>=self.threshold:
                self.stats["hits"]+=1; return e["resp"],True
        resp=client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()
        self.cache.append({"emb":qe,"resp":resp})
        self.stats["misses"]+=1; return resp,False

c=SemanticCache(threshold=0.90)
for q in ["What is machine learning?","What is ML?","Explain machine learning",
         "What is deep learning?","Tell me about deep learning","How does Python work?"]:
    ans,hit=c.query(q)
    print(f"  {'⚡ HIT (free!)' if hit else '🔄 MISS (LLM)'} '{q}'")
h,m=c.stats["hits"],c.stats["misses"]
print(f"\n  Hit rate: {h}/{h+m} ({h/(h+m)*100:.0f}%) -> {h} free calls")

# Output: (representative run)
#   🔄 MISS (LLM) 'What is machine learning?'
#   ⚡ HIT (free!) 'What is ML?'
#   ⚡ HIT (free!) 'Explain machine learning'
#   🔄 MISS (LLM) 'What is deep learning?'
#   ⚡ HIT (free!) 'Tell me about deep learning'
#   🔄 MISS (LLM) 'How does Python work?'
#
#   Hit rate: 3/6 (50%) -> 3 free calls
# Tradeoff: every point of threshold you trade for hit rate is a point of
# wrong-answer risk - 'What is MLOps?' can false-match 'What is ML?'. The
# animation below has that exact failure. Review borderline hits by hand.

- `_embed()` - turns each question into a vector via the embedding model (a few paise per call - the cache's only running cost). Embeddings put "What is ML?" and "What is machine learning?" nearly on top of each other in vector space.

- the cosine-similarity loop - compares the new question against every cached one; a linear scan is fine for a demo, a vector index (Module 8 territory) replaces it in production.

- `threshold=0.90` - the risk dial. Higher = safer but fewer free hits; lower = more hits and more wrong-order servings. There is no universal number: tune it on YOUR traffic and hand-review borderline hits.

- the miss path - pays for one real generation, then files the answer so the next paraphrase is free.

Read it as: *vectorize the question, look for a close-enough neighbour, serve or cook accordingly.*

#### 💡 What just happened

⚡ What just happened?"What is ML?" matched "What is machine learning?" (92% similarity). Cache hit = zero cost + instant. **Production hit rates: 40-70%. A 60% hit rate = 60% cost AND latency reduction.**

- **Scene 1 (hits and misses):** each incoming question shows its similarity meter against the best cached entry. Above the line: HIT - instant and free. Below: MISS - a paid LLM call whose answer joins the cache. Notice: the hit-rate and money-saved counters as paraphrases pile up.

- **Scene 2 (the threshold dial):** lower the threshold and hits climb - until "What is MLOps?" false-matches "What is ML?" and the cache serves the WRONG answer with full confidence. Do: find the threshold where your appetite for savings meets your appetite for risk.

Controls: Play / Reset / Speed (Slow / Normal / Fast); threshold presets; scene buttons or arrow keys.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Prompt Compression - Fewer Tokens, Same Info

### Prompt Compression - Fewer Tokens, Same Info

Remove filler, abbreviate, restructure. Same answer, 65% fewer tokens.

Caching removes repeat questions; compression shrinks what is left. Every request re-bills your instructions, so a bloated system prompt is a subscription you never meant to buy.

> 📧 **Analogy**
>
> **The telegram era priced every word - so people wrote like it.** "ARRIVING TUESDAY 3PM STATION" carries everything "I just wanted to let you know that I will most likely be arriving..." does, at a fifth of the cost. Token pricing is telegram pricing: the model does not need your politeness, your company history, or three adjectives per noun - it needs role, constraints, facts. Compress like a telegram clerk: meaning stays, filler pays.

**📝 `04_compression.py`** - *Compression*

In [ ]:
# PROMPT COMPRESSION - Same info, fewer tokens
import tiktoken, re
enc=tiktoken.get_encoding("cl100k_base")
def ct(t): return len(enc.encode(t))

bloated="""You are a very helpful and friendly customer support assistant for our 
company. Our company is called Netsetos and we are an educational technology company 
based in the beautiful city of Hyderabad, India. We specialize in providing high-quality 
courses in the field of Artificial Intelligence and Machine Learning. When a customer 
asks you a question, please provide a helpful, detailed, and accurate response. Please 
note that our refund policy allows full refunds within 7 days of purchase. After 7 days, 
we offer a 50% refund up to 30 days. After 30 days, no refunds are available.
Customer question: What is your refund policy?"""

compressed="""Role: Netsetos support (edtech, Hyderabad). Scope: AI/ML courses.
Refund: 7d=100%, 8-30d=50%, 30d+=none.
Q: What is your refund policy?"""

print(f"  Original:   {ct(bloated)} tokens")
print(f"  Compressed: {ct(compressed)} tokens")
print(f"  Reduction:  {(1-ct(compressed)/ct(bloated))*100:.0f}%")
print(f"  Same answer. {(1-ct(compressed)/ct(bloated))*100:.0f}% cheaper per call.")
print(f"  At 100K calls/day on GPT-5.4 ($2.50/M in): saves ~${(ct(bloated)-ct(compressed))*100000*2.5/1e6:.0f}/day")

# Output: (real run - tiktoken is exactly right for THIS job: comparing formats)
#   Original:   161 tokens
#   Compressed: 51 tokens
#   Reduction:  68%
#   Same answer. 68% cheaper per call.
#   At 100K calls/day on GPT-5.4 ($2.50/M in): saves ~$27/day

## Operate at Scale: Observability, Stacking, and Budgets

---

## 🎯 Concept 5: Cost Observability - Helicone, Langfuse, Provider Dashboards

### Cost Observability - Helicone, Langfuse, Provider Dashboards

You can't optimize what you can't measure. One-line Helicone. Trace-level Langfuse.

The four core techniques are installed. Operating them at scale starts with visibility: a bill you can only see as one number is a bill you cannot argue with, assign, or alarm on.

> 🧾 **Analogy**
>
> **Lump-sum bill vs itemized bill.** "Food: ₹12,000" starts a family argument; an itemized bill - who ordered the lobster, which night, which table - settles it in seconds. Helicone and Langfuse are the itemizers: every request tagged with feature, user, and model, so "why did April double?" has an answer ("the new summarizer feature, premium model, 40% of spend") instead of a suspect list.

| Tool | Integration | Cost Tracking | Free Tier | Best For |
|---|---|---|---|---|
| Helicone | 1-line URL change | Auto per request | 10K req/mo | Fastest to deploy |
| Langfuse | SDK decorator | Trace-level attribution | 50K units/mo | Agent workflow tracing |
| LiteLLM | Proxy server | Per user/team/key | Free (open-source) | Budget enforcement |
| OpenAI Dashboard | Native | By model/project | Included | Quick overview |
| Anthropic Admin API | REST API | 1-min granularity | Included | Programmatic monitoring |

> 💡 **Info**
>
> ⚠️ OpenAI removed hard budget limits in 2025
> OpenAI project budgets are now **soft alerts only** - requests continue uninterrupted when budget is exceeded. Teams relying on native OpenAI controls for cost protection are unprotected. Use LiteLLM, Portkey, or Bifrost for hard budget enforcement with graduated thresholds: alert at 50%, throttle at 80%, downgrade models at 90%, block at 100%.

#### 💡 What just happened

What just happened?**Helicone** (YC W23, ~5.7K stars; acquired by Mintlify, Mar 2026) is the fastest to deploy - change your OpenAI base URL to `oai.helicone.ai/v1`, add one header, done. Every request is logged with cost, latency, tokens, and custom metadata. **Langfuse** (acquired by ClickHouse, Jan 2026) provides deeper trace-level attribution - cost per trace, user, session, and feature. **Critical:** OpenAI's native budget controls are now soft-only. Enforce hard budgets at the gateway layer (LiteLLM/Portkey). Dashboard must track 8 metrics: cost per request, per user, per feature, per conversation, cache hit ratio, model distribution, trend, and error rate. Tracing and evaluation on top of these dashboards is Lesson 7.6 (Observability with Langfuse).

---

## 🎯 Concept 6: Token Optimization - Six Techniques That Compound to 50-80% Savings

### Token Optimization - Six Techniques That Compound to 50-80% Savings

max_tokens, system prompt, few-shot, stop sequences, output format, embedding models.

Observability shows WHERE money goes; these six techniques shrink each line item. None is dramatic alone - the point is that they compound, and they all attack the expensive side of the meter: output tokens.

> 🧳 **Analogy**
>
> **Packing for an airline that charges by the kilo.** Six habits of the frequent flyer: a hard-side case that physically cannot overpack (`max_tokens`), rolling clothes instead of folding (JSON over prose), leaving the "maybe" items home (2 few-shot examples, not 8), a packing list that stops you at socks (stop sequences), wearing the heavy coat onto the plane (move constants into cached prefixes), and picking the budget airline for cargo that does not care (cheap embedding models). No single trick saves the fee - together they halve it.

> ✅ **Info**
>
> 📈 Six token optimization techniques
> - **max_tokens:** Set explicit caps. Classification: 50. Chat: 150-300. Code: 500-1000. Reduces average output by 40%, saving 20-30% total cost. Caution: reasoning models bill invisible thinking tokens - budget 3-5× expected output.
> - **System prompt compression:** 2,000-token prompt at 1M requests costs $5,000/mo on GPT-4o. Compress to 500 tokens = $1,250/mo. Strip verbose role descriptions to telegraphic instructions. Treat as production code.
> - **Few-shot selection:** Diminishing returns after 2-3 examples. Sweet spot: 2 examples for most tasks. Newer reasoning models (o1, o3) perform worse with few-shot. Fine-tuning breakeven: ~50M tokens/month.
> - **Stop sequences:** Prevent generation beyond useful response. Can save 20-50% of output tokens. Output tokens cost 2-5× input tokens - targeting the most expensive part of the bill.
> - **Output format:** JSON uses 60% fewer tokens than prose for same data. TSV uses 50% of JSON's tokens. OpenAI structured output saves ~44% total tokens. Anthropic now has NATIVE structured outputs (messages.parse / output_config.format) - the old tool_use workaround and its token overhead are obsolete.
> - **Embedding models:** 6-20× cost variation. OpenAI text-embedding-3-small: $0.02/MTok. Self-hosted NV-Embed-v2: ~$0.001/MTok (beats all commercial APIs at 72.3 MTEB). Gemini: free tier available.

#### 💡 What just happened

What just happened? Six techniques that **compound to 50-80% savings** without quality degradation. System prompt compression has outsized impact because it's paid on every request. Stop sequences target output tokens (the expensive side). Structured output format choices matter: JSON vs TSV vs TOON can save 40-60%. Embedding model selection offers the widest cost range - self-hosted NV-Embed beats commercial APIs in both quality AND cost. Apply all six together: the savings multiply, not add.

---

## 🎯 Concept 7: Prompt Caching & Batch API - Stack to 95% Reduction

### Prompt Caching & Batch API - Stack to 95% Reduction

Anthropic 90% off cached reads. OpenAI 90% auto. Batch 50% off. Combined: 95%.

Everything so far reduced what you send. The providers themselves offer the two biggest levers left: pay a tenth for input they have already seen, and half price for output you can wait for. They multiply.

> 🎫 **Analogy**
>
> **A season ticket plus the night tariff.** The commuter pass makes your daily route nearly free after the first purchase - that is prompt caching: the system prompt you send every request bills at ~10% once the provider has it cached (mind the one-time pass fee: cache WRITES cost 1.25x on Anthropic). The night tariff is the batch API: same electricity, half price, delivered off-peak within 24 hours. A commuter who uses both pays a twentieth of the tourist - 0.1 x 0.5 = 0.05.

| Technique | Provider | Discount | How It Works |
|---|---|---|---|
| Prompt Caching | Anthropic | 90% off reads | Explicit cache_control, 5-min TTL, 1.25× write cost |
| OpenAI | 90% off (auto) | Automatic; cached input bills at 10% of the input price |  |
| Google | 90% off (2.5+) | Implicit (auto) for Gemini 2.5+, configurable TTL |  |
| Batch API | OpenAI | 50% off | JSONL upload, results within 24hrs |
| Anthropic | 50% off | Stacks with cache for 95% total reduction |  |
| Combined | Anthropic | 95% off | Cache 0.1× × Batch 0.5× = 0.05× ($3.00→$0.15/M) |

#### 💡 What just happened

What just happened? Prompt caching and batch API are the **two most powerful cost levers**. They're multiplicative: Anthropic cache (90% off) × batch (50% off) = 95% reduction. Claude Sonnet drops from $3.00/M to $0.15/M input. A customer service bot dropped from **$50K→$15K/month** with 73% cache hit rate alone. Latency improves up to 85% - a 100K-token prompt drops from 11.5s to 2.4s. **Rule:** any non-real-time task should use batch API. Any task with repeated system prompts should use caching. One 7.2 callback matters here: a stream cancelled mid-flight may never report its usage - meter cancelled streams from accumulated text or your cache-hit math drifts.

---

## 🎯 Concept 8: Tiered Routing & Self-Hosting - RouteLLM, vLLM, Break-Even

### Tiered Routing & Self-Hosting - RouteLLM, vLLM, Break-Even

Route 70% to cheap models. RouteLLM saves 85%. Self-hosting at 50M+ tokens/month.

Two-tier routing was the training wheels; production fleets run three or four tiers, and at some volume the "own vs rent" question appears. Both decisions hinge on one metric that is not on any pricing page.

> 🚛 **Analogy**
>
> **Running a delivery fleet.** Scooters for envelopes (nano models), vans for furniture (mid-tier), a truck for the rare full-house move (frontier). Buying your own truck (self-hosting on vLLM) only beats renting when it hauls every day - roughly 50M+ tokens a month - and someone must be paid to service it. The paradox every fleet manager learns: the cheapest scooter per kilometre, if it breaks down and redelivers, costs more per PARCEL than the van - price the successful delivery, not the fuel.

| Tier | Traffic % | Models | Cost ($/M input) | Use Case |
|---|---|---|---|---|
| Budget | 60-70% | GPT-5.4-nano, DeepSeek V4-Flash, Gemini Flash-Lite | $0.05-0.27 | Classification, FAQ, extraction |
| Mid-tier | 20-25% | GPT-5.4-mini, Claude Haiku 4.5, Gemini 2.5 Flash | $0.30-1.00 | Summarization, generation |
| Premium | 5-10% | Claude Sonnet 4.6, GPT-5.4, Gemini 2.5 Pro | $1.25-3.00 | Complex reasoning, synthesis |
| Frontier | 1-2% | Claude Opus 4.8, GPT-5.5, reasoning modes | $5.00-30.00 | Research, novel problems |

> 📦 **Info**
>
> 🚨 The cost paradox: cheapest per-token ≠ cheapest per-task
> **Reported case study (Gitar, Feb 2026 - verify before quoting):** Switched from Claude Sonnet ($3/MTok) to Kimi K2.5 ($0.60/MTok) - 5× cheaper per token. Costs went up. One PR review burned 500K+ tokens in infinite retry loops because the cheaper model had worse self-correction. Worse, switching providers destroyed prompt cache hits. **The correct metric is cost-per-successful-output (CPSO):** total spend ÷ outputs requiring no correction. A model with 95% success rate needs ~1,053 calls for 1,000 usable outputs. A 40% success rate needs ~2,500. Always measure CPSO, not cost-per-token.

#### 💡 What just happened

What just happened?**RouteLLM** (LMSYS/Berkeley) maintains 95% GPT-4 quality with 85% cost reduction by routing to cheap models when possible. AWS Bedrock offers built-in Intelligent Prompt Routing. **Self-hosting** (vLLM on H100) breaks even at ~50M tokens/month vs premium APIs, but requires 10-20 hrs/month maintenance. Against budget APIs like DeepSeek at $0.27/M, self-hosting rarely wins. **The cost paradox** is the most important lesson: Gitar's 5× cheaper model cost MORE because of retries, cache invalidation, and lower quality. Measure cost-per-successful-output, not cost-per-token. Fine-tuning a small model becomes the cheaper play at roughly 50M+ tokens/month - that break-even analysis is Module 9 (Fine-Tuning); fleet-scale FinOps lives in Module 13 (Cost & Performance).

---

## 🎯 Concept 9: Budget Enforcement - Dashboards, Alerts, Chargebacks

### Budget Enforcement - Dashboards, Alerts, Chargebacks

Graduated thresholds. Per-team budgets. FinOps for AI. Cost allocation.

Everything above optimizes the expected bill. This step is about the unexpected one: the retry loop at 3 AM, the leaked key, the feature that went viral. Alerts inform; only enforcement protects.

> 🔋 **Analogy**
>
> **Postpaid alerts vs a prepaid meter.** A postpaid electricity plan happily emails you "usage is high!" while the AC runs all summer - that is a provider budget alert (OpenAI's limits are now soft-only: requests keep processing past 100%). A prepaid meter physically cuts out. Production budgets need the prepaid model at the gateway - with graduated steps first, like dimming the lights before a blackout: alert at 50%, throttle at 80%, downgrade models at 90%, block at 100%.

> ✅ **Info**
>
> 💰 Production budget enforcement architecture
> - **Graduated thresholds:** Alert at 50% → throttle at 80% → downgrade models at 90% → block at 100%. Never hard-block without warning - degrade gracefully.
> - **Gateway budget enforcement:** LiteLLM supports per-key, per-user, per-team hard caps (returns HTTP 429). Bifrost: 4-tier hierarchy (Customer→Team→Key→Provider). Portkey: built-in spend caps per virtual key.
> - **8 dashboard metrics:** Cost per request, per user, per feature, per conversation, cache hit ratio, model distribution, cost trend, error/retry rate.
> - **Chargeback models:** Per-token pass-through (most transparent), per-request flat fee (predictable), tiered pricing (Basic/Premium), credit pool (teams buy credits).
> - **A/B testing costs:** Cost-per-successful-output per variant. Langfuse prompt versioning with quality+cost per variant. Larger sample sizes needed (LLM outputs are stochastic).

#### 💡 What just happened

What just happened? Budget enforcement is the final safety net. **Never rely on provider-native controls** - OpenAI's are soft-only. Implement graduated thresholds at the gateway layer: alert→throttle→downgrade→block. Cost allocation requires tagging every request with team/project/feature metadata via virtual API keys. **FinOps for AI** is an emerging discipline - the FinOps Foundation has published reference architectures for GenAI cost tracking. For internal platforms: chargeback models make teams cost-conscious. For A/B testing: always compare cost-per-successful-output, not just quality scores.

---

## 🎯 Concept 10: India Cost Planning - GST, Indic Token Tax, ₹ Budgets

### India Cost Planning - GST, Indic Token Tax, ₹ Budgets

18% GST reverse charge. Hindi costs 63% more tokens. Sarvam AI: free LLMs.

The final layer is the one your finance team sees first: the difference between the sticker price on a pricing page and the landed price on an Indian invoice - taxes, forex, and a surcharge most builders never notice until they ship in Hindi.

> 🚘 **Analogy**
>
> **Showroom price vs on-road price.** No one drives out of an Indian showroom at the sticker: registration, insurance, and GST turn ₹10 lakh into ₹12 lakh at the gate. Foreign LLM APIs work the same - sticker, then +18% GST, then the card's forex markup. And shipping in Hindi is like a luggage surcharge on every trip: the same sentence packs into ~63% more tokens on English-trained tokenizers. The domestic model (Sarvam) skips the import duty and the surcharge - or pre-translate and fly the English fare.

| Cost Factor | Impact | Mitigation |
|---|---|---|
| GST (18% IGST) | +18% on foreign APIs | GST-registered: claim ITC (net: cash-flow only). Use AWS/Azure India for INR invoicing. |
| Forex markup | +1.5-3.5% on cards | Wise/Niyo virtual cards. AWS/Azure/GCP India accounts for ₹ billing. |
| Indic tokenization | Hindi: +63% tokens. Tamil: +100% | Sarvam AI tokenizer (1× English efficiency). Or pre-translate to English. |
| Exchange rate | ₹93/USD (Apr 2026) | Budget in USD, convert to ₹ monthly. Hedge with 3-month contracts. |

> 💡 **Info**
>
> 🇮🇳 Indian startup LLM budget benchmarks
> - **Pre-seed/MVP:** ₹5,000-25,000/month ($50-$300) - free tiers + budget models
> - **Seed:** ₹25,000-2,00,000/month ($300-$2,500) - 1K-10K users
> - **Series A:** ₹2,00,000-10,00,000/month ($2,500-$12,000) - 40K+ users
> - **Sarvam AI:** open-weights LLMs (105B, 30B - free to self-host, Apache 2.0); hosted API from ~₹4/M input + ₹16/M output. STT at ₹30/hr. All billing in ₹.
> - **Krutrim Cloud:** Llama 4 hosting at ₹7-17/M tokens. Claims 25× cheaper GPU instances vs AWS.

#### 💡 What just happened

What just happened? India adds **20-60% to LLM sticker prices** through GST, forex, and the Indic tokenization tax. The **tokenization penalty is real and quantifiable:** Hindi costs 63% more tokens than English on GPT-4o, and 4-5× more on Llama 3.3. At 100M Hindi words/month, that's an extra ₹2 lakh/year. **Three mitigations:** Sarvam AI's tokenizer (eliminates the penalty), pre-translate to English, or choose GPT-4o/Gemini over Llama (71% better Indic efficiency). Indian startups need 2-3× more aggressive optimization because Indian SaaS ARPU is 1/5th to 1/10th of US ARPU.

## The Full Cost Stack

> 📦 **Info**
>
> 4 layers combined = 80-95% savings
> - **Cache (Layer 1):** 40-70% answered from cache. Cost: $0. Latency: <50ms.
> - **Compression (Layer 2):** 30-65% fewer tokens per call.
> - **Cheap-first (Layer 3):** 70-90% served by Flash ($0.10/M) not GPT-4o ($2.50/M).
> - **Monitoring (Layer 4):** Track, alert, kill switch for runaway costs.

| Strategy | Savings | Effort | Speed |
|---|---|---|---|
| Model routing | 40-60% | Low | Immediate |
| Response caching | 30-50% | Low | Immediate |
| Prompt compression | 20-40% | Medium | 1-2 weeks |
| Batch API | 50% | Low | Non-realtime |
| Fine-tuned small | 60-80% | High | Weeks |

## Key Takeaways

> ✅ **Info**
>
> What you learned
> - **Count every token.** CostTracker measures and projects daily/monthly spend.
> - **Cheap-first-escalate:** 70-90% served by Flash. Quality check costs almost nothing.
> - **Semantic caching:** 40-70% hit rate = 40-70% free calls. "What is ML?" = "What is machine learning?"
> - **Prompt compression:** Remove filler, abbreviate, restructure. 30-65% fewer tokens.
> - **Combined:** ₹5L/month → ₹30K/month. 94% savings. Same quality.
> - **Observe and enforce:** Helicone/Langfuse for measurement; provider budgets are soft alerts only - hard caps (alert 50% → throttle 80% → downgrade 90% → block 100%) live in your gateway.
> - **Stack the discounts:** prompt caching (~90% off reads on Anthropic, OpenAI, and Gemini) x batch (50%) multiply to ~95% on eligible traffic - and measure CPSO, not cost-per-token.
> - **India landed price:** sticker + GST 18% + forex + the Indic token tax; mitigate with ₹-billed regional endpoints, Sarvam's tokenizer, or pre-translation.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Cost Engineering- Every Token Has a Price**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-7_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-7.3.html` - regenerate this notebook whenever the source HTML is updated.*
